In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestClassifier

In [2]:
# Definindo os caminho da nova pasta no PC pessoal
# O caminho usa '../../' pois o notebook está em scripts/Redução de Dimensionalidade/
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/'

nome_dados_treinamento = 'Sem Redução de Dimensionalidade/lycos_cicids2017_treinamento.csv'
nome_dados_teste = 'Sem Redução de Dimensionalidade/lycos_cicids2017_teste.csv'

nome_dados_treinamento_reduzidos = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv'
nome_dados_teste_reduzidos = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv'

In [3]:
# 1. Carregando o input a partir dos CSVs de dados JÁ TRATADOS
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")

display(df_treino.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1286248, 80) | Teste: (551250, 80)


,src_port,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,pkt_len_min,pkt_len_mean,pkt_len_var,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,Label
0,0.951110,0.000809,0.122137,0.493151,0.000504,0.123909,0.003948,0.031768,0.030108,0.000050,...,0.000000,0.000000,0.000000,0.000026,0.000005,1.495151e-07,0.000003,0.000000,0.000000,benign
1,0.766598,0.006760,0.038168,0.235429,0.025492,0.041303,0.001249,0.000000,0.003345,0.000009,...,0.000000,0.000000,0.000000,0.000009,0.000007,0.000000e+00,0.000002,0.003922,0.021423,benign
2,0.814466,0.006760,0.038168,0.512447,0.979619,0.111518,0.057131,0.000000,0.066731,0.004397,...,0.000004,0.000069,0.000009,0.000228,0.000030,2.463439e-06,0.000021,0.445572,0.647110,benign
3,0.074983,0.000809,0.122137,0.240826,0.000002,0.123909,0.010475,0.022790,0.061262,0.000639,...,0.000000,0.000000,0.000000,0.000037,0.000009,7.933453e-07,0.000007,0.000000,0.000000,benign
4,0.847898,0.006760,0.038168,0.235855,0.070290,0.149890,0.175020,0.000000,0.392759,0.041693,...,0.000387,0.000566,0.000418,0.000322,0.000094,6.456254e-05,0.000086,0.445572,0.176773,benign


In [4]:
# 2. Removendo características identificadoras/enviesadas antes da seleção por MDI
# No dataset tratado atual, o campo identificador/enviesado ainda presente é a porta de destino.
# A coluna duplicada de Fwd Header Length também é removida para reduzir redundância.
nomes_enviesados = {
    'Flow ID',
    'Source IP',
    'Source Port',
    'Destination IP',
    'Destination Port',
    'Protocol',
    'Timestamp',
    'timestamp',
    'Fwd Header Length.1'
}

colunas_para_remover = []
fwd_header_length_vistos = 0

for coluna in df_treino.columns:
    nome_normalizado = coluna.strip().lower()

    if coluna in nomes_enviesados:
        colunas_para_remover.append(coluna)

    if nome_normalizado == 'fwd header length':
        fwd_header_length_vistos += 1
        if fwd_header_length_vistos > 1:
            colunas_para_remover.append(coluna)

    if nome_normalizado.startswith('fwd header length.'):
        colunas_para_remover.append(coluna)

colunas_para_remover = [coluna for coluna in dict.fromkeys(colunas_para_remover) if coluna != 'Label']
colunas_treino_existentes = [coluna for coluna in colunas_para_remover if coluna in df_treino.columns]
colunas_teste_existentes = [coluna for coluna in colunas_para_remover if coluna in df_teste.columns]

df_treino = df_treino.drop(columns=colunas_treino_existentes)
df_teste = df_teste.drop(columns=colunas_teste_existentes)

print('Características removidas para mitigar overfitting:', colunas_treino_existentes)
print(f'Dimensões após remoção - Treino: {df_treino.shape} | Teste: {df_teste.shape}')

# 3. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

Características removidas para mitigar overfitting: ['timestamp']
Dimensões após remoção - Treino: (1286248, 79) | Teste: (551250, 79)


In [5]:
# 4. Aplicando Seleção de Características MDI (Mean Decrease in Impurity) com Random Forest Balanceado
print("\nIniciando o treinamento do Random Forest para extração de features (MDI)...")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced')

inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()

print(f"Treinamento RF concluído! Tempo extração: {(fim_fs - inicio_fs):.4f} segundos.")

importancias = rf_fs.feature_importances_
features = X_treino.columns

df_importancias = pd.DataFrame({'Feature': features, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)

print("\nQuantidade total de features:", len(features))

# Selecionando as 51 features mais importantes (conforme sugerido na literatura por Neto e Gomes)
top_51_features = df_importancias.head(51)['Feature'].tolist()

quantidade_de_features_descartadas = len(features) - len(top_51_features)
print("Quantidade de features selecionadas (top 51):", len(top_51_features))
print("Quantidade de features descartadas:", quantidade_de_features_descartadas)

# Mostra qual foi o corte de importância para as features selecionadas
importancia_corte = df_importancias[df_importancias['Feature'] == top_51_features[-1]]['Importancia'].values[0]
print(f"Importância mínima para entrar no top 51: {importancia_corte:.6f}")
display(df_importancias.tail(quantidade_de_features_descartadas + 5))

print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))


Iniciando o treinamento do Random Forest para extração de features (MDI)...
Treinamento RF concluído! Tempo extração: 33.3403 segundos.

Quantidade total de features: 78
Quantidade de features selecionadas (top 51): 51
Quantidade de features descartadas: 27
Importância mínima para entrar no top 51: 0.005018


,Feature,Importancia
38,fwd_iat_min,6.755297e-03
54,flag_SYN,6.706613e-03
12,fwd_pkt_per_s,5.392410e-03
13,bwd_pkt_per_s,5.330714e-03
51,idle_min,5.017751e-03
55,flag_fin,4.479901e-03
6,pkt_len_min,4.167894e-03
11,pkt_per_s,3.670549e-03
50,idle_max,3.365798e-03
56,flag_rst,3.119371e-03



Top 10 features mais importantes:


,Feature,Importancia
0,src_port,0.041654
1,dst_port,0.039278
25,bwd_pkt_len_max,0.039001
3,flow_duration,0.034381
27,bwd_pkt_len_mean,0.033061
5,pkt_len_max,0.032568
16,fwd_pkt_len_max,0.030279
4,down_up_ratio,0.027519
20,fwd_pkt_hdr_len_tot,0.027230
19,fwd_pkt_len_std,0.026624


In [6]:
# 5. Reduzindo a dimensionalidade dos conjuntos de treino e teste
df_treino_reduzido = X_treino[top_51_features].copy()
df_treino_reduzido['Label'] = y_treino.values

df_teste_reduzido = X_teste[top_51_features].copy()
df_teste_reduzido['Label'] = y_teste.values

print(f"Novas dimensões - Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")

# Salvando os datasets reduzidos em CSV para uso posterior
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_dados_treinamento_reduzidos, index=False)
print(f"Dataset de treino reduzido salvo em: {caminho_pasta_tratado + nome_dados_treinamento_reduzidos}")

df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_dados_teste_reduzidos, index=False)
print(f"Dataset de teste reduzido salvo em: {caminho_pasta_tratado + nome_dados_teste_reduzidos}")

Novas dimensões - Treino: (1286248, 52) | Teste: (551250, 52)
Dataset de treino reduzido salvo em: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv
Dataset de teste reduzido salvo em: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv
